In [109]:
import importlib
import sys
sys.path.insert(0, '../')

import os
from pathlib import Path
import json

from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.environment import RoutingEnv
from src.agent import QLearningAgent
from config.graphs import (
    GRAPH_PLANT,
    NODE_ROLES_PLANT,
    DEFAULT_START_CANDIDATES,
    DEFAULT_PICKUP_CANDIDATES,
    DEFAULT_DROP_CANDIDATES,
)

# Paths
PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
LOGS_DIR = RESULTS_DIR / "logs"
MODELS_DIR = RESULTS_DIR / "models"
EVAL_DIR = RESULTS_DIR / "evaluation"
PLOTS_DIR = RESULTS_DIR / "plots"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [110]:
if 'src.environment' in sys.modules:
    importlib.reload(sys.modules['src.environment'])
    from src.environment import RoutingEnv
    print("✓ Módulo environment.py recargado")
else:
    from src.environment import RoutingEnv
    print("✓ Módulo environment.py cargado por primera vez")

✓ Módulo environment.py recargado


In [111]:
import random

def set_global_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

set_global_seeds(42)

In [112]:
from datetime import datetime
from src.agent import QLearningAgent
from src.environment import RoutingEnv

REWARD_PARAMS = {
    "r_step": -1.0,
    "r_queue_factor": -0.3,
    "r_task_completion": 50.0,
    "r_failure": -30.0,
}
MAX_STEPS = 100

ALPHA = 0.1
GAMMA = 0.95
EPSILON_START = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995

NUM_EPISODES = 3000

def run_reference_training(seed: int = 42):
    set_global_seeds(seed)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = LOGS_DIR / f"train_log_{timestamp}.csv"
    model_path = MODELS_DIR / f"q_table_final_{timestamp}.pkl"

    env = RoutingEnv(
        graph=GRAPH_PLANT,
        node_roles=NODE_ROLES_PLANT,
        reward_params=REWARD_PARAMS,
        max_steps=MAX_STEPS,
        start_candidates=DEFAULT_START_CANDIDATES,
        pickup_candidates=DEFAULT_PICKUP_CANDIDATES,
        drop_candidates=DEFAULT_DROP_CANDIDATES,
        queue_sampling="uniform",
        seed=seed,
        penalize_revisits=True,
        revisit_penalty_factor=-2.0,
    )

    agent = QLearningAgent(
        alpha=ALPHA,
        gamma=GAMMA,
        epsilon=EPSILON_START,
        epsilon_min=EPSILON_MIN,
        epsilon_decay=EPSILON_DECAY,
        seed=seed,
    )

    rows = []

    for episode in range(1, NUM_EPISODES + 1):
        state = env.reset()
        done = False
        total_reward = 0.0
        steps = 0

        while not done:
            valid_actions = env.get_valid_actions()
            action = agent.select_action(state, valid_actions)
            next_state, reward, done, info = env.step(action)

            if not done:
                next_valid_actions = env.get_valid_actions()
            else:
                next_valid_actions = []

            agent.update(state, action, reward, next_state, next_valid_actions, done)

            state = next_state
            total_reward += reward
            steps += 1

        agent.decay_epsilon()
        termination_reason = info.get("termination_reason", "unknown")

        rows.append((episode, total_reward, steps, termination_reason, agent.epsilon))

    # Save log
    df_log = pd.DataFrame(rows, columns=["episode", "total_reward", "steps", "termination_reason", "epsilon"])
    df_log.to_csv(log_path, index=False)

    # Save model
    agent.save(str(model_path))

    return log_path, model_path

log_path, model_path = run_reference_training(seed=42)
print("Log guardado en:", log_path)
print("Modelo guardado en:", model_path)

Agent saved to /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl
Log guardado en: /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/logs/train_log_20251230_204138.csv
Modelo guardado en: /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl


In [113]:
log_files = sorted(LOGS_DIR.glob("train_log_*.csv"))
model_files = sorted(MODELS_DIR.glob("q_table_final_*.pkl"))

assert log_files, "No training logs found"
assert model_files, "No final models found"

log_path = log_files[-1]
model_path = model_files[-1]

log_path, model_path

(PosixPath('/home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/logs/train_log_20251230_204138.csv'),
 PosixPath('/home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl'))

In [114]:
from pathlib import Path
import importlib
import experiments.evaluate as evaluate

importlib.reload(evaluate)

# Usar el model_path que ya tienes en el notebook (del entrenamiento previo)
print("Evaluando modelo:", model_path)

all_results = evaluate.evaluate_agent(
    model_path=str(model_path),
    queue_levels=evaluate.QUEUE_LEVELS_TO_TEST,
    num_episodes=evaluate.NUM_EVAL_EPISODES,
)

# Guardar manualmente el JSON, igual que en main()
EVAL_DIR.mkdir(parents=True, exist_ok=True)
eval_json_path = EVAL_DIR / "evaluation_results.json"
with open(eval_json_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("✓ Evaluación completada, resultados en:", eval_json_path)

Evaluando modelo: /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl
Q-LEARNING AGENT - EVALUATION

Loading model: /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl
Agent loaded from /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl
 Q-table size: 898 states
 Current epsilon: 0.0500

Evaluation configuration:
  Episodes per queue level: 100
  Queue levels to test: [0, 1, 2]
  Policy: Greedy (ε=0)
  Random seed: 999

Evaluating on queue_level=0...
  Avg reward: 42.30 ± 1.71
  Avg steps: 7.7 ± 1.7
  Success rate: 100.0% (100/100)

Evaluating on queue_level=1...
  Avg reward: 39.78 ± 2.50
  Avg steps: 7.9 ± 1.9
  Success rate: 100.0% (100/100)

Evaluating on queue_level=2...
  Avg reward: 37.84 ± 2.60
  Avg steps: 7.6 ± 1.6
  Success rate: 100.0% (100/100)

✓ Evaluación completada, resultados en: /home/jpmt/GitHub/Master/04-IntroductionML/Tra

In [115]:
eval_json_path = EVAL_DIR / "evaluation_results.json"
assert eval_json_path.exists(), f"{eval_json_path} not found"

with open(eval_json_path, "r") as f:
    eval_results = json.load(f)

# Keys can be str → Change to int
eval_results = {int(k): v for k, v in eval_results.items()}
eval_results

{0: {'queue_level': 0,
  'num_episodes': 100,
  'avg_reward': 42.3,
  'std_reward': 1.7117242768623688,
  'avg_steps': 7.7,
  'std_steps': 1.711724276862369,
  'success_count': 100,
  'failure_count': 0,
  'success_rate': 1.0,
  'min_reward': 38.0,
  'max_reward': 44.0,
  'min_steps': 6,
  'max_steps': 12},
 1: {'queue_level': 1,
  'num_episodes': 100,
  'avg_reward': 39.782,
  'std_reward': 2.5007350919279725,
  'avg_steps': 7.86,
  'std_steps': 1.9236423784061318,
  'success_count': 100,
  'failure_count': 0,
  'success_rate': 1.0,
  'min_reward': 34.4,
  'max_reward': 42.2,
  'min_steps': 6,
  'max_steps': 12},
 2: {'queue_level': 2,
  'num_episodes': 100,
  'avg_reward': 37.84,
  'std_reward': 2.5996922894835066,
  'avg_steps': 7.6,
  'std_steps': 1.624807680927192,
  'success_count': 100,
  'failure_count': 0,
  'success_rate': 1.0,
  'min_reward': 30.8,
  'max_reward': 40.4,
  'min_steps': 6,
  'max_steps': 12}}

In [116]:
queue_levels = [0, 1, 2]
rows = []

for q in queue_levels:
    if q not in eval_results:
        continue
    r = eval_results[q]
    rows.append({
        "queue_level": q,
        "episodes": r["num_episodes"],
        "avg_reward": r["avg_reward"],
        "std_reward": r["std_reward"],
        "avg_steps": r["avg_steps"],
        "std_steps": r["std_steps"],
        "success_rate": r["success_rate"],
        "success_count": r["success_count"],
        "failure_count": r["failure_count"],
    })

df_metrics = pd.DataFrame(rows)
df_metrics

,queue_level,episodes,avg_reward,std_reward,avg_steps,std_steps,success_rate,success_count,failure_count
0,0,100,42.300,1.711724,7.70,1.711724,1.0,100,0
1,1,100,39.782,2.500735,7.86,1.923642,1.0,100,0
2,2,100,37.840,2.599692,7.60,1.624808,1.0,100,0


In [117]:
def run_greedy_episode(
    model_path: Path,
    queue_level: int = 2,
    seed: int = 123,
    verbose: bool = True,
):
    """
    Ejecuta un episodio greedy con cola fija.
    Devuelve una lista de dicts con el recorrido.
    """
    set_global_seeds(seed)

    # Cargar agente
    agent = QLearningAgent()
    agent.load(str(model_path))

    # Política greedy
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0

    # Entorno con cola fija
    env = RoutingEnv(
        graph=GRAPH_PLANT,
        node_roles=NODE_ROLES_PLANT,
        reward_params=REWARD_PARAMS,
        max_steps=MAX_STEPS,
        start_candidates=DEFAULT_START_CANDIDATES,
        pickup_candidates=DEFAULT_PICKUP_CANDIDATES,
        drop_candidates=DEFAULT_DROP_CANDIDATES,
        queue_sampling="fixed",
        fixed_queue_level=queue_level,
        seed=seed,
        penalize_revisits=True,
        revisit_penalty_factor=-2.0,
    )

    traj = []

    state = env.reset()
    done = False
    episode_reward = 0.0
    step = 0

    if verbose:
        print(f"Initial job: start={env.current_job['start']}, "
              f"pickup={env.current_job['pickup']}, drop={env.current_job['drop']}")

    while not done:
        location, battery, queue, load_state, pickup, drop = state
        valid_actions = env.get_valid_actions()
        action = agent.select_action(state, valid_actions)

        next_state, reward, done, info = env.step(action)

        traj.append({
            "step": step,
            "state": state,
            "action": action,
            "next_state": next_state,
            "reward": reward,
            "termination_reason": info.get("termination_reason", None) if done else None,
        })

        if verbose:
            print(f"t={step:02d} | loc={location:2d}, load={load_state}, "
                  f"action→{action:2d}, reward={reward:6.1f}")

        state = next_state
        episode_reward += reward
        step += 1

    if verbose:
        print(f"\nEpisode finished in {step} steps, total_reward={episode_reward:.1f}, "
              f"reason={info.get('termination_reason', 'unknown')}")

    # Restaurar epsilon
    agent.epsilon = original_epsilon

    return traj

In [118]:
from datetime import datetime

traj = run_greedy_episode(model_path=model_path, queue_level=2, seed=123, verbose=True)

uses_shortcut = any(
    (s["state"][0] == 2 and s["action"] == 7) or
    (s["state"][0] == 7 and s["action"] == 2)
    for s in traj
)
uses_shortcut

def check_shortcut_usage(model_path: Path, queue_level: int = 2, n_runs: int = 20):
    count = 0
    for i in range(n_runs):
        traj = run_greedy_episode(model_path, queue_level=queue_level, seed=100 + i, verbose=False)
        if any(
            (s["state"][0] == 2 and s["action"] == 7) or
            (s["state"][0] == 7 and s["action"] == 2)
            for s in traj
        ):
            count += 1
    print(f"Shortcut 2↔7 used in {count}/{n_runs} runs")

def save_trajectory_to_csv(traj, save_dir: Path, prefix: str = "trajectory"):
    save_dir.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(traj)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = save_dir / f"{prefix}_{timestamp}.csv"
    df.to_csv(path, index=False)
    print(f"Trayectoria guardada en: {path}")
    return path

def summarize_trajectories(model_path: Path, queue_level: int = 2, n_runs: int = 20):
    stats = []
    for i in range(n_runs):
        seed = 100 + i
        traj = run_greedy_episode(model_path, queue_level=queue_level, seed=seed, verbose=False)
        uses_shortcut = any(
            (s["state"][0] == 2 and s["action"] == 7) or
            (s["state"][0] == 7 and s["action"] == 2)
            for s in traj
        )
        termination = traj[-1]["termination_reason"]
        steps = len(traj)
        total_reward = sum(t["reward"] for t in traj)
        stats.append({
            "seed": seed,
            "queue_level": queue_level,
            "uses_shortcut": uses_shortcut,
            "termination_reason": termination,
            "steps": steps,
            "total_reward": total_reward,
        })
    df_stats = pd.DataFrame(stats)
    return df_stats

df_traj_stats = summarize_trajectories(model_path, queue_level=2, n_runs=20)
df_traj_stats

check_shortcut_usage(model_path, queue_level=2, n_runs=20)

TRAJ_DIR = RESULTS_DIR / "trajectories"
traj_path = save_trajectory_to_csv(traj, TRAJ_DIR, prefix="queue2_seed123")
traj_path

Agent loaded from /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl
 Q-table size: 898 states
 Current epsilon: 0.0500
Initial job: start=0, pickup=14, drop=15
t=00 | loc= 0, load=0, action→ 1, reward=  -1.6
t=01 | loc= 1, load=0, action→ 2, reward=  -1.6
t=02 | loc= 2, load=0, action→ 3, reward=  -1.6
t=03 | loc= 3, load=0, action→ 4, reward=  -1.6
t=04 | loc= 4, load=0, action→ 5, reward=  -1.6
t=05 | loc= 5, load=0, action→14, reward=  -1.6
t=06 | loc=14, load=1, action→ 5, reward=  -1.6
t=07 | loc= 5, load=1, action→ 6, reward=  -1.6
t=08 | loc= 6, load=1, action→ 7, reward=  -1.6
t=09 | loc= 7, load=1, action→15, reward=  48.4

Episode finished in 10 steps, total_reward=34.0, reason=success
Agent loaded from /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/models/q_table_final_20251230_204138.pkl
 Q-table size: 898 states
 Current epsilon: 0.0500
Agent loaded from /home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results

PosixPath('/home/jpmt/GitHub/Master/04-IntroductionML/Trabajo/results/trajectories/queue2_seed123_20251230_204142.csv')